# Gold 2: embed the chunks

Task `embed` of the job `indexing`. Fills `gold_children.embedding` for every chunk that has none, with
`ai_query` on the embedding endpoint. Only `embed_text` (the prefix line with the filing and section, then the
child's text) is embedded; `parent_text` is not.

**Each chunk is paid for once.** The vectors are stored in the Delta table, not only in the index, so the
Vector Search index can be deleted and built again (`test_retrieval` does that every run) without embedding
anything again. A rerun only embeds the rows still missing.

**Save points**, as in silver: `batch_size` rows at a time. The answers of a batch are collected first and then
written, so each paid call runs once. A failed row (`errorMessage`) stays empty and is tried again by the next
batch or run. A batch that stores nothing stops the run (the endpoint is failing: run again later).

The dimension of the vectors depends on the model (1024 for `databricks-gte-large-en`); `test_retrieval` needs
the same value (`embedding_dimension`).

In [ ]:
import sys

sys.path.insert(0, "../common")
from delta.tables import DeltaTable

from gold_tables import create_gold_tables
from names import Names

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("gold_schema", "finance_bench_gold")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en")
dbutils.widgets.text("batch_size", "2000")

w = dbutils.widgets.get
n = Names(w("catalog"), w("bronze_schema"), w("silver_schema"), w("gold_schema"))
endpoint = w("embedding_endpoint")
batch_size = int(w("batch_size"))

create_gold_tables(spark, n)

In [ ]:
def embed_batch():
    """Embed up to batch_size chunks that have no vector yet. Returns (stored, failed)."""
    rows = spark.sql(f"""
    SELECT chunk_id, ai_query('{endpoint}', embed_text, failOnError => false) AS r
    FROM (SELECT chunk_id, embed_text FROM {n.children} WHERE embedding IS NULL LIMIT {batch_size})
    """).collect()
    good = [(r.chunk_id, [float(x) for x in r.r.result]) for r in rows if not r.r.errorMessage and r.r.result]
    if good:
        vectors = spark.createDataFrame(good, "chunk_id string, embedding array<float>")
        (DeltaTable.forName(spark, n.children).alias("t")
         .merge(vectors.alias("s"), "t.chunk_id = s.chunk_id")
         .whenMatchedUpdate(set={"embedding": "s.embedding"}).execute())
    failed = len(rows) - len(good)
    if failed:
        print("  first error:", next(r.r.errorMessage for r in rows if r.r.errorMessage or not r.r.result))
    return len(good), failed


total = 0
while True:
    stored, failed = embed_batch()
    total += stored
    print(f"{stored} chunks embedded, {failed} failed")
    if stored == 0:
        break
print(f"Embedded in this run: {total}.")

In [ ]:
display(spark.sql(f"""
SELECT strategy, count(*) AS chunks, count(embedding) AS embedded, count(*) - count(embedding) AS missing,
       max(size(embedding)) AS dimension
FROM {n.children} GROUP BY strategy ORDER BY strategy"""))